# COSC726 project demo - MCP Ambulance Handoff Gateway (HL7 FHIR R4)

Omair Gibreel. Runtime > Run all. Part A needs no GPU and no API key; all patients are synthetic.

Part B (optional) installs Ollama and runs the agent with the open-weight `qwen2.5:7b-instruct` model. It needs a T4 GPU runtime (Runtime > Change runtime type).

In [ ]:
!git clone -q https://github.com/comrade1996/cosc726-omairgibreel.git
%cd cosc726-omairgibreel
!python -m pip install -q -r requirements.txt
!python --version

## A1. Tests: pipeline, security gates, MCP stdio and HTTP end-to-end

In [ ]:
!python -m pytest -q tests

## A2. The agent prepares a complete handoff over MCP. The simulated clinician approves.

In [ ]:
!python -m project.agent.host --case AMB-001 --approve auto

## A3. Safe block: an incomplete record is escalated, not guessed

In [ ]:
!python -m project.agent.host --case AMB-019 --approve auto

## A4. Controlled failure and recovery: a hallucinated tool and an invented blood pressure are both caught

In [ ]:
!python -m project.agent.host --case AMB-001 --model faulty --approve auto

## A5. Prompt injection in the narrative, and a cross-crew access attempt

In [ ]:
!python -m project.agent.host --case AMB-035 --approve auto | tail -8
!python -m project.agent.host --case AMB-001 --crew CREW-B --approve auto | tail -6

## A6. Second client over Streamable HTTP (bearer auth and Origin checks)

In [ ]:
!python -m project.http_client --spawn --case AMB-002

## A7. Full evaluation: 40 cases x 4 configurations, gated on `evals/thresholds.json`

In [ ]:
!python -m evals.run_eval --check

## B. Optional: a real open-weight LLM as the agent (T4 GPU)

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull qwen2.5:7b-instruct > /dev/null && echo pulled

In [ ]:
import os
os.environ.update(LLM_BASE_URL='http://127.0.0.1:11434/v1', LLM_MODEL='qwen2.5:7b-instruct')
!python -m project.agent.host --case AMB-001 --model openai --approve auto

In [ ]:
!python -m evals.run_llm_eval --cases AMB-001,AMB-019,AMB-029,AMB-035 --skip-rq3